### Fetch Transcript

In [53]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
from sentence_transformers import SentenceTransformer
import spacy
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import numpy as np
from typing import List, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
load_dotenv()

True

In [2]:
# fetch video id from url
def fetch_video_id(url:str)-> str:
    try:
        pattern = re.compile(
            r"(?:youtube\.com\/watch\?v=)([a-zA-Z0-9_-]{11})"
        )
        match = re.search(pattern, url)
        return match.group(1) if match else None
    except Exception as e:
        print(f"[INFO] No transcript found. Error: {e}")
        raise

In [3]:
url = "https://www.youtube.com/watch?v=GlYgs6v2YfU"
video_id = fetch_video_id(url)
print(video_id)

GlYgs6v2YfU


In [4]:
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id = video_id)


In [5]:
fetched_transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="There's a kind of wild paper from 2002 called Language Trees and Zipping,", start=0.0, duration=4.071), FetchedTranscriptSnippet(text='which shows how you can find structure between languages just by using very general file', start=4.071, duration=4.964), FetchedTranscriptSnippet(text='compression.', start=9.035, duration=0.725), FetchedTranscriptSnippet(text="For example, let's say I give you a bunch of text documents in many different languages,", start=10.46, duration=4.162), FetchedTranscriptSnippet(text='and your goal is to automatically cluster them by language.', start=14.622, duration=2.838), FetchedTranscriptSnippet(text='And actually, you can aim even higher than that.', start=18.04, duration=1.82), FetchedTranscriptSnippet(text='Suppose you also want to recover which languages are most closely related to each', start=20.1, duration=4.084), FetchedTranscriptSnippet(text='other in a way that lets you rediscover the tre

In [ ]:
transcript:list = []
for snipet in fetched_transcript:
    transcript.append(snipet.text)

In [7]:
text = " ".join(transcript)

In [8]:
text

"There's a kind of wild paper from 2002 called Language Trees and Zipping, which shows how you can find structure between languages just by using very general file compression. For example, let's say I give you a bunch of text documents in many different languages, and your goal is to automatically cluster them by language. And actually, you can aim even higher than that. Suppose you also want to recover which languages are most closely related to each other in a way that lets you rediscover the tree of shared lineage between them, and all you can do is write a function that processes the text in each one. There's no pre-baked knowledge of linguistics. Now keep in mind, this is almost a quarter century ago, so nothing like modern language models is available. What's crazy is how the authors showed that you can do this just by using gzip, basically the same operation that you might use on your computer to make files a bit smaller. Here's the basic idea. Given two documents, A and B, app

### Chunking

In [9]:
# linguistic preprocessings to get proper sentences

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
doc

There's a kind of wild paper from 2002 called Language Trees and Zipping, which shows how you can find structure between languages just by using very general file compression. For example, let's say I give you a bunch of text documents in many different languages, and your goal is to automatically cluster them by language. And actually, you can aim even higher than that. Suppose you also want to recover which languages are most closely related to each other in a way that lets you rediscover the tree of shared lineage between them, and all you can do is write a function that processes the text in each one. There's no pre-baked knowledge of linguistics. Now keep in mind, this is almost a quarter century ago, so nothing like modern language models is available. What's crazy is how the authors showed that you can do this just by using gzip, basically the same operation that you might use on your computer to make files a bit smaller. Here's the basic idea. Given two documents, A and B, appe

In [10]:
sentences = [sent.text for sent in doc.sents]

In [11]:
text_splitter = RecursiveCharacterTextSplitter(separators = ["\n\n", "\n", ". ", "? ", ", ", "! ", " ", ""], chunk_size = 700, chunk_overlap = 100, length_function = len)
text_splitter.split_text("".join(sentences))

["There's a kind of wild paper from 2002 called Language Trees and Zipping, which shows how you can find structure between languages just by using very general file compression.For example, let's say I give you a bunch of text documents in many different languages, and your goal is to automatically cluster them by language.And actually, you can aim even higher than that.Suppose you also want to recover which languages are most closely related to each other in a way that lets you rediscover the tree of shared lineage between them, and all you can do is write a function that processes the text in each one.There's no pre-baked knowledge of linguistics.Now keep in mind",
 ", this is almost a quarter century ago, so nothing like modern language models is available.What's crazy is how the authors showed that you can do this just by using gzip, basically the same operation that you might use on your computer to make files a bit smaller.Here's the basic idea.Given two documents, A and B, appen

#### Semantic chunking

In [25]:
len(sentences)

255

In [36]:
def semantic_chunking(sentences:List[str], similarity_threshold:float = 0.20, max_tokens:int = 500, overlap:int = 1):
    embedder = OpenAIEmbeddings(model = "text-embedding-3-large")
    embeddings = embedder.embed_documents(sentences)
    chunks = []
    current_chunk = []
    current_tokens = 0

    for i, sentence in enumerate(sentences):
        sentence_tokens = len(sentence.split())

        if not current_chunk:
            current_chunk.append(sentence)
            current_tokens += sentence_tokens 
            continue

        # calculate similarity score between sentence and previous sentence
        sim = np.dot(embeddings[i], embeddings[i-1]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i-1]))

        if sim < similarity_threshold and current_tokens > max_tokens:
            chunks.append(" ".join(current_chunk))

            current_chunk = current_chunk[-overlap:] if overlap > 0 else []
            current_tokens = sum(len(s.split()) for s in current_chunk)

        current_chunk.append(sentence)
        current_tokens += sentence_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [37]:
chunks = semantic_chunking(sentences=sentences, similarity_threshold=0.15, max_tokens=500)

In [83]:
# savings chunks for experimentation
import pickle
with open('test-chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

In [38]:
len(chunks)

10

In [39]:
# generate embeddings of chunks

embedder = OpenAIEmbeddings(model = "text-embedding-3-large")
embeddings = embedder.embed_documents(chunks)

In [51]:
embeddings = np.array(embeddings)

### Vectorstore

In [54]:
embeddings.shape

(10, 3072)

In [56]:
# create FAISS index

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

In [ ]:
# add embeddings
index.add(embeddings)

In [58]:
# save the index 
faiss.write_index(index, "faiss.index")


In [59]:
# load faiss index
loaded_index = faiss.read_index("faiss.index")

In [76]:
# testing new query
query = "What is 3b1b Talent?"
query_embedding = embedder.embed_documents([query])

query_embedding = np.array(query_embedding)

In [77]:
query_embedding.shape

(1, 3072)

In [78]:
D, I = loaded_index.search(query_embedding, k= 2)

In [79]:
I

array([[8, 9]])

In [81]:
chunks[8]

"Looking back at that My Name is Blank example, instead of having to see many thousands or many millions of examples in the data before your loss represents the cross-entropy against a reasonable distribution of possible names, you can get that same effect just on a single example, comparing it against the good model's prediction. I think now is a good time to zoom out and look over everything we've built up. In the first half, in the context of compression, cross-entropy arose very naturally just by asking how well a compression scheme optimized for one setting performs in another. The mild surprise is that this idea turns out to be much more generally useful, as a way to measure how different patterns in one setting are from those in another. This is what makes it such a common tool for loss functions in modern machine learning. All that said, as stated so far, the way that the formula showed up in the second half, for us, feels pretty different from the way that it showed up in the 